# SafeTread - ResNet50 Binary Classifier
## Training on Google Colab GPU

**Dataset**: Good vs Defective Tyres (Binary Classification)

**Model**: ResNet50 Transfer Learning + Fine-tuning

**Expected Accuracy**: 95%+

In [ ]:
import tensorflow as tf
import os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import shutil

print('='*60)
print('SafeTread Model Training - ResNet50 Binary Classification')
print('='*60)

# ===== 0) TensorFlow & GPU Check =====
print('\n0️⃣ TensorFlow Setup')
print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ===== 0.5) Mount Google Drive =====
print('\n0️⃣.5️⃣ Mounting Google Drive...')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✓ Google Drive Mounted')

In [ ]:
# ===== 1) Dataset Configuration & Verification =====
print('\n1️⃣ Dataset Configuration')

DATASET_DIR = '/content/drive/MyDrive/Tyre dataset'
SPLITS = ['train', 'val', 'test']
CLASSES = ['Good', 'Defective']
ALLOWED_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def count_images(split_dir):
    """Count images in each class folder"""
    counts = defaultdict(int)
    for cls in CLASSES:
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(ALLOWED_EXTS)]
        counts[cls] += len(files)
    return counts

def verify_dataset():
    """Verify dataset structure and count images"""
    print('\n=== Dataset Structure Check ===')
    total_images = 0
    for split in SPLITS:
        split_dir = os.path.join(DATASET_DIR, split)
        if not os.path.isdir(split_dir):
            print(f'❌ Missing split folder: {split_dir}')
            continue
        counts = count_images(split_dir)
        split_total = sum(counts.values())
        total_images += split_total
        
        print(f'\n📁 Split: {split}')
        for cls in CLASSES:
            print(f'   {cls}: {counts[cls]} images')
            if counts[cls] == 0:
                print(f'   ⚠️  EMPTY folder detected!')
        
        if all(counts[cls] > 0 for cls in CLASSES):
            ratio = max(counts.values()) / min(counts.values())
            if ratio > 1.5:
                print(f'   ⚠️  Class imbalance (ratio: {ratio:.2f})')
    
    print(f'\n✓ Total images across all splits: {total_images}')
    return total_images > 0

# Create directory structure
print('\n=== Creating Dataset Directories ===')
for split in SPLITS:
    split_path = os.path.join(DATASET_DIR, split)
    os.makedirs(split_path, exist_ok=True)
    for cls in CLASSES:
        class_path = os.path.join(split_path, cls)
        os.makedirs(class_path, exist_ok=True)
        print(f'✓ {split}/{cls}')

# Verify dataset
has_images = verify_dataset()

if not has_images:
    print('\n⚠️  WARNING: No images found!')
    print('Please upload images to Google Drive:')
    print(f'  {DATASET_DIR}/train/Good/')
    print(f'  {DATASET_DIR}/train/Defective/')
    print(f'  (and similarly for val/ and test/)')

In [ ]:
if has_images:
    # ===== 2) Load Data =====
    print('\n2️⃣ Loading Data...')
    
    train_dir = os.path.join(DATASET_DIR, 'train')
    val_dir = os.path.join(DATASET_DIR, 'val')
    test_dir = os.path.join(DATASET_DIR, 'test')
    
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASSES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=42
    )
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASSES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASSES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    
    print('✓ Data loaded successfully')
    
    # ===== 3) Data Augmentation =====
    print('\n3️⃣ Setting Up Data Augmentation...')
    
    def augment_fn(images, labels):
        images = tf.image.random_flip_left_right(images)
        images = tf.image.random_rotation(images, 0.08)
        images = tf.image.random_zoom(images, [0.9, 1.1])
        images = tf.image.random_contrast(images, 0.9, 1.1)
        return images, labels
    
    train_ds = train_ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)
    print('✓ Data augmentation configured')

In [ ]:
if has_images:
    # ===== 4) Build ResNet50 Model (Sequential - CLEAN SERIALIZATION) =====
    print('\n4️⃣ Building ResNet50 Model...')
    
    from tensorflow.keras.applications import ResNet50
    from tensorflow.keras.applications.resnet import preprocess_input
    from tensorflow.keras import layers
    
    base_model = ResNet50(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    
    model = tf.keras.Sequential([
        layers.Input(shape=IMG_SIZE + (3,)),
        layers.Lambda(lambda x: preprocess_input(x)),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    print('✓ Model created')
    print('\nModel Architecture:')
    print(f'  Input: {IMG_SIZE} (224x224 RGB images)')
    print(f'  Base: ResNet50 (frozen, 23.5M parameters)')
    print(f'  Top: BatchNorm → Dropout(0.3) → Dense(256) → Dense(1, sigmoid)')
    print(f'  Total trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')
    
    # ===== 5) Compile Model =====
    print('\n5️⃣ Compiling Model...')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    print('✓ Model compiled')
    
    # ===== 6) Callbacks =====
    print('\n6️⃣ Setting Up Callbacks...')
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            'best_model.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]
    print('✓ Callbacks configured')

In [ ]:
if has_images:
    # ===== 7) Train Model =====
    print('\n7️⃣ Training (25 epochs)...')
    print('=' * 60)
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        callbacks=callbacks,
        verbose=1
    )
    
    print('=' * 60)
    print('✓ Training complete')

In [ ]:
if has_images:
    # ===== 8) Evaluate on Test Set =====
    print('\n8️⃣ Evaluating on Test Set...')
    
    y_true = []
    y_pred = []
    
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        preds_binary = (preds > 0.5).astype(int).flatten()
        y_true.extend(labels.numpy().astype(int))
        y_pred.extend(preds_binary)
    
    cm = confusion_matrix(y_true, y_pred)
    print('\n🎯 Confusion Matrix:')
    print(cm)
    print('\nGood (0) = columns, Defective (1) = rows')
    
    print('\n📊 Classification Report:')
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    
    # Plot training history
    print('\n📈 Plotting Training History...')
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if has_images:
    # ===== 9) Fine-Tuning =====
    print('\n9️⃣ Fine-Tuning (8 epochs)...')
    print('=' * 60)
    
    # Unfreeze last 20 layers of ResNet50
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        callbacks=callbacks,
        verbose=1
    )
    
    print('=' * 60)
    print('✓ Fine-tuning complete')

In [ ]:
if has_images:
    # ===== 10) Save Model =====
    print('\n🔟 Saving Model...')
    model.save('best_model_finetuned.h5')
    print('✓ Model saved as best_model_finetuned.h5')
    
    # ===== 11) Download Model & Save to Drive =====
    print('\n1️⃣1️⃣ Downloading Model...')
    from google.colab import files
    files.download('best_model_finetuned.h5')
    print('✓ Download started (check Colab downloads)')
    
    # Also save to Google Drive as backup
    import shutil
    shutil.copy('best_model_finetuned.h5', '/content/drive/MyDrive/best_model_finetuned.h5')
    print('✓ Also saved to Google Drive: MyDrive/best_model_finetuned.h5')
    
    print('\n' + '='*60)
    print('✅ TRAINING COMPLETE!')
    print('='*60)
    print('\nNext Steps:')
    print('1. Move best_model_finetuned.h5 to: SafeTread/ml/models/')
    print('2. Restart Flask backend: python app.py')
    print('3. Test predictions with frontend')
    print('='*60)
else:
    print('\n❌ TRAINING SKIPPED - No images found')
    print('Please upload your images to Google Drive first')